In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits

def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool

def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    #selector_proton = 'TightKMProtonSelection'
    #selector_pion = 'TightKMPionMicroSelection'
    #print(f"Now trying to create a mask with {selector_proton}")
    #print(f"Now trying to create a mask with {selector_pion}")
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2

def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector='SuperLooseKMProtonSelection',
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)

    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)

    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1

    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector='LooseKMProtonSelection'):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")

pidChoiceDict = {}

for pidChoice in [1, 157, 202, 112, 256, 121]:
    if pidChoice == 1:
        myPMask = "SuperLooseKMProtonSelection"
        myPiMask = "SuperLooseKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    elif pidChoice == 157:
        myPMask = "VeryLooseKMProtonSelection"
        myPiMask = "VeryLooseKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    elif pidChoice == 202:
        myPMask = "VeryLooseKMProtonSelection"
        myPiMask = "TightKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    elif pidChoice == 112:
        myPMask = "VeryLooseKMProtonSelection"
        myPiMask = "LooseKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    elif pidChoice == 256:
        myPMask = "LooseKMProtonSelection"
        myPiMask = "VeryLooseKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    elif pidChoice == 121:
        myPMask = "SuperLooseKMProtonSelection"
        myPiMask = "SuperTightKMPionMicroSelection"
        pidChoiceDict[pidChoice] = (myPMask,myPiMask)
    
    for [maxSigCut,minSigCut] in [[20,10],[10,10], [20,20],[40,10],[30,10]]:
        
        for sig in [False]:
            bkgE=0
            dataB = True
            for spChoice in ['998','1005']:

                indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

                mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)
                
                spmask = (data['spmode']==spChoice)

                mes = data['BpostFitMes']
                de = data['BpostFitDeltaE']

                meslo = 5.2
                mesloS = 5.27
                meshiS =5.3
                DeltaElo = -0.2
                DeltaEhi = 0.2
                DeltaEloS = -0.07
                DeltaEhiS = 0.07

                mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 
                
                mask_basic = mask_pid & (mes>meslo) & (de>DeltaElo) & (de<DeltaEhi)
                
                mask_fit = mask_pid & (mes>meslo)  & (de>DeltaElo) & (de<DeltaEhi) & (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))
                
                mask_fit_sig = mask_pid & (mes>mesloS) & (mes<meshiS) & (de>DeltaEloS) & (de<DeltaEhiS)


                mask_fit = mask_fit[spmask]
                mask_fit_sig = mask_fit_sig[spmask]

                if sig:
                    mask_fit_choice = mask_fit_sig
                else:
                    mask_fit_choice = mask_fit


                lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit_choice]
                lamuncmass = data[spmask]['Lambda0_unc_Mass'][mask_fit_choice]
                numBCan = data[spmask]['nB']

                lamuncmassUNCUT = data[spmask]['Lambda0_unc_Mass']






                lammass_world_average = 1.115683
                width = 0.003
                lo = lammass_world_average - width
                hi = lammass_world_average + width
                    

                mask_fl = (lamconflsig>minSigCut)
                mask_upper_fl = (lamconflsig>maxSigCut)

                
                flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))


                bCanMask = (numBCan==1)

                lamuncmassMask = (lamuncmass>lo)&(lamuncmass<hi)


                vetoMask = apply_antibaryon_veto(data, pps, myPMask)
                vetoMask = vetoMask[spmask][bCanMask]

                m = lamuncmass

                m = m[flightSigMask&lamuncmassMask]
                m = m[bCanMask]
                m = m[vetoMask]

                #print(spChoice, ak.sum(ak.num(m)))

                if not dataB:
                    print("PID CHOICE: ", pidChoiceDict[pidChoice])
                    print("Max,Min Sig Cut:", maxSigCut,minSigCut)
                    if sig:
                        print("!Signal Region! (5.27 < mES < 5.3 and | Delta E | < 0.07)")
                    else:
                        print("!Sideband Region! (5.2 < mES and | Delta E | < 0.2)")
                    print("SP Mode:", spChoice)
                    if ak.sum(ak.num(lamuncmassUNCUT)) == 0:
                        print("Efficiency: lamuncmassUNCUT=0, DivBy0 Error")
                    else:
                        print("Efficiency:", ak.sum(ak.num(m))/ak.sum(ak.num(lamuncmassUNCUT)))
                    print("---------------------------------------------------------")

                if spChoice == '998':
                    bkgE += (ak.sum(ak.num(m)))*(424.3/1719)
                elif spChoice == '1005':
                    bkgE += (ak.sum(ak.num(m)))*(424.3/868.3)
            if dataB:

                indices_and_mapsD = get_info_for_PID_masks(blindD, verbosity=0)

                mask_bool_protonL1D, mask_bool_pionL1D,mask_bool_protonL2D, mask_bool_pionL2D = PID_masks(indices_and_mapsD, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)
                mesD = blindD['BpostFitMes']
                deD = blindD['BpostFitDeltaE']

                meslo = 5.2
                mesloS = 5.27
                meshiS =5.3
                DeltaElo = -0.2
                DeltaEhi = 0.2
                DeltaEloS = -0.07
                DeltaEhiS = 0.07

                mask_pidD = mask_bool_pionL1D & mask_bool_protonL1D & mask_bool_pionL2D & mask_bool_protonL2D
                
                mask_basicD = mask_pidD & (mesD>meslo) & (deD>DeltaElo) & (deD<DeltaEhi)
                
                mask_fitD = mask_pidD & (mesD>meslo)  & (deD>DeltaElo) & (deD<DeltaEhi) & (((mesD<mesloS) | (mesD>meshiS)) | ((deD<DeltaEloS) | (deD>DeltaEhiS)))
                
                lamconflsigD = blindD['Lambda0postFitFlightSignificance'][mask_fitD]
                lamuncmassD = blindD['Lambda0_unc_Mass'][mask_fitD]
                numBCanD = blindD['nB']

                mask_flD = (lamconflsigD>minSigCut)
                mask_upper_flD = (lamconflsigD>maxSigCut)
                
                flightSigMaskD = (ak.all(mask_flD,axis=1))&(ak.any(mask_upper_flD, axis=1))

                bCanMaskD = (numBCanD==1)

                lamuncmassMaskD = (lamuncmassD>lo)&(lamuncmassD<hi)

                vetoMaskD = apply_antibaryon_veto(blindD, pps, myPMask)
                vetoMaskD = vetoMaskD[bCanMaskD]

                mD = lamuncmassD

                mD = mD[flightSigMaskD&lamuncmassMaskD]
                mD = mD[bCanMaskD]
                mD = mD[vetoMaskD]

                myMD = list(val for val in mD if len(val)>0)
                dataC = len(myMD)

                print("PID CHOICE: ", pidChoiceDict[pidChoice])
                print("Max,Min Sig Cut:", maxSigCut,minSigCut)
                if sig:
                    print("!Signal Region! (5.27 < mES < 5.3 and | Delta E | < 0.07)")
                else:
                    print("!Sideband Region! (5.2 < mES and | Delta E | < 0.2)")
            if sig:
                print("Weighted and summed bkg events in signal region:", bkgE, "\n")
            else:
                print("Weighted and summed bkg events in sideband region:", bkgE)
                if dataB:
                    print("Data events in sideband region:", dataC ,"\n")
                else:
                    print()
            print("#########################################################")
            print("#########################################################\n")


            

            """
            mesP = mes[spmask][mask_fit]

            mesP = mesP[flightSigMask&lamuncmassMask]

            mesP = mesP[bCanMask]

            mesP = mesP[vetoMask]


            deP = de[spmask][mask_fit]

            deP = deP[flightSigMask&lamuncmassMask]

            deP = deP[bCanMask]

            deP = deP[vetoMask]







            
            print("PID CHOICE: ", pidChoice)
            print("Max,Min Sig Cut:", maxSigCut,minSigCut)
            print("SP Mode:", spChoice)
            plt.figure(figsize = (8,6))
            plt.hist2d(
                    ak.to_numpy(ak.flatten(mesP)), 
                    ak.to_numpy(ak.flatten(deP)),
                    bins=100, 
                    range=[[5.0,5.4], [-0.25,0.25]], 
                    cmap='viridis'
                )

            plt.colorbar(label='Candidates per bin')
            plt.xlabel("m_ES")
            plt.ylabel("deltaE")
            plt.title('2D m_ES vs DeltaE correlation')
            """
            """
            # 2. Create a 2D Histogram (highly recommended for large particle physics datasets)
            if True:
                plt.figure(figsize=(8, 6))
                plt.hist2d(
                    x_combined, 
                    y_combined, 
                    bins=100, 
                    range=[[lo-1.05*width, hi+1.05*width], [lo-1.05*width, hi+1.05*width]], 
                    cmap='viridis'
                )

                # 3. Add labels and a colorbar
                plt.colorbar(label='Candidates per bin')
                plt.xlabel(r'$Lambda$ Candidate 1 Mass [GeV/$c^2$]')
                plt.ylabel(r'$Lambda$ Candidate 2 Mass [GeV/$c^2$]')
                plt.title('2D Mass Correlation')

                plt.show()
            """
            """
            nsig = len(m[s5Mask]) - b1 - b0

            nbkg = b1+b0
            npeak = len(m[s5Mask])
            nall = len(lamuncmass[lamfl_basic>=0]) #?????? Should this be events with 2 lambda, events with enough fltsig, or all events?

            #nall = len(ak.flatten(m[lamfl_basic>=0], axis=None))


            #npeak = len(ak.flatten(m[mask], axis=None))
            #nbkglo = len(ak.flatten(m[mask_lo_sideband], axis=None))
            #nbkghi = len(ak.flatten(m[mask_hi_sideband], axis=None))
            #nbkg = (nbkglo + nbkghi)/2.0
            #nsig = npeak - nbkg

            #print(f"nall: {nall}     npeak: {npeak}   nbkg: {nbkg}     nsig: {nsig}")
            peaks.append(npeak)
            sigs.append(nsig)
            bkgs.append(nbkg)
            cuts.append(i)
            allentries.append(nall)
            tags.append(flight_tag)

            row = int(i/ncols)
            col = i%ncols
            
            plt.sca(axes1[row][col])
                
            plt.hist(ak.flatten(lamuncmass, axis=None),bins=100,range=(1.105, 1.125), label=f'flcut>{i:.1f}')
            plt.hist(ak.flatten(m, axis=None),      bins=100,range=(1.105, 1.125))
            #plt.hist(ak.flatten(m[mask_lo_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
            #plt.hist(ak.flatten(m[mask_hi_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
            plt.legend()

            #plt.sca(axes2[row][col])
            if row == nrows-1:
                axes1[row][col].set_xlabel(r'$Lambda^0$ mass [GeV/c$^2$]')
                #axes2[row][col].set_xlabel(r'$Lambda^0$ flight len [cm]')

            fllo, flhi = 0,30
            if flight_tag=='lampostfitsig':
                fllo, flhi = 0,100

            #plt.hist(ak.flatten(lamconflsig, axis=None),         bins=100,range=(fllo, flhi))
            #plt.hist(ak.flatten(lamconflsig[mask_fl], axis=None),bins=100,range=(fllo, flhi))

            #fig1.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)
            #fig2.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)
            
            #fig1.subplots_adjust(0,0,1,1,0,0)
            #fig2.subplots_adjust(0,0,1,1,0,0)

            #plt.figure(fig1)
            #plt.suptitle(f'{flight_tag}')
            #plt.figure(fig2)
            #plt.suptitle(f'{flight_tag}')
            
            #fig1.tight_layout()
            #fig2.tight_layout()
            

            sigs = np.array(sigs)
            bkgs = np.array(bkgs)
            peaks = np.array(peaks)
            cuts = np.array(cuts)
            allentries = np.array(allentries)

            pcts = sigs/sigs[0]
            bkg_under_peak = peaks - sigs
            pct_bkg_under_peak = bkg_under_peak / bkg_under_peak[0]

            S = sigs
            B = bkg_under_peak
            scale = 1
            fom = scale*S/np.sqrt(scale*S + scale*B)
            
            max_fom = max(fom)
            idx_max = fom.tolist().index(max_fom)
            print(valu, "/", myPMask, myPiMask)
            print(f"idx_max: {idx_max}  max cuts: {cuts[idx_max]:.2f} max fom: {fom[idx_max]:.2f}   pcts: {pcts[idx_max]:.2f}")


            plt.figure(figsize=(12, 8))
            plt.subplot(3,2,1)
            plt.plot(cuts, peaks,'o')
            plt.ylabel('# under peak')
            plt.xlabel(f'Cut on flight sig value')

            plt.subplot(3,2,2)
            plt.plot(cuts, sigs,'o')
            #plt.ylabel('# of signal in peak (# in peak - # est background')
            plt.ylabel('# of signal in peak')
            plt.xlabel(f'Cut on flight sig value')

            plt.subplot(3,2,3)
            plt.plot(cuts, bkgs,'o')
            plt.ylabel('# est background')
            plt.xlabel(f'Cut on flight sig value')
            
            plt.subplot(3,2,4)
            plt.plot(cuts, pcts,'o')
            plt.ylim(0.1,1.1)
            plt.ylabel('% sig remaining')
            plt.xlabel(f'Cut on flight sig value')
            
            plt.subplot(3,2,5)
            plt.plot(cuts, pct_bkg_under_peak,'o')
            #plt.ylim(0.7,1.1)
            plt.ylabel('% bkg under peak')
            plt.xlabel(f'Cut on flight sig value')
            
            # Naive significance
            # Multiply by 10 if we are only doing Run 1
            plt.subplot(3,2,6)
            #plt.plot(cuts, 10*sigs/np.sqrt(10*bkg_under_peak),'o')
            plt.plot(cuts, fom,'o')
            plt.ylabel(r'$S/sqrt{ S + B }$')
            plt.xlabel(f'Cut on flight sig value')

            plt.suptitle(f'MYLABEL')

            plt.tight_layout()
            """
            #plt.legend() 
            #plt.show()